# 04 — Lung Ultrasound Data Inspection & Model Readiness

Mục tiêu: kiểm tra dữ liệu siêu âm phổi, nhãn, chất lượng ảnh và khả năng chia tập theo bệnh nhân. Không huấn luyện nếu chưa có dữ liệu ảnh có nhãn.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
LUNG_RAW = PROJECT_DIR / "data" / "LUNG_ULTRASOUND" / "raw"
LUNG_PROCESSED = PROJECT_DIR / "data" / "LUNG_ULTRASOUND" / "processed"
IMG_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

print("Project:", PROJECT_DIR)
print("Raw folder exists:", LUNG_RAW.exists())
images = [p for p in LUNG_RAW.rglob("*") if p.is_file() and p.suffix.lower() in IMG_EXTS]
print("Number of image files:", len(images))

In [ ]:
if not images:
    print("NO LUNG ULTRASOUND IMAGES FOUND.")
    print("Add labeled images under data/LUNG_ULTRASOUND/raw/<class_name>/, for example:")
    print("  raw/normal/*.png")
    print("  raw/abnormal/*.png")
    print("Do not create synthetic images or labels. Training is blocked until real labeled images are added.")
else:
    rows = []
    for p in images:
        try:
            with Image.open(p) as im:
                rows.append({"path": str(p), "class_folder": p.parent.name, "width": im.width, "height": im.height, "mode": im.mode, "format": im.format})
        except Exception as e:
            rows.append({"path": str(p), "class_folder": p.parent.name, "error": str(e)})
    lung_df = pd.DataFrame(rows)
    display(lung_df.head())
    print("Images by folder:")
    display(lung_df["class_folder"].value_counts())

In [ ]:
if images:
    valid = lung_df.dropna(subset=["width", "height"])
    print("Valid images:", len(valid), "/", len(lung_df))
    display(valid[["width", "height"]].describe())
    valid["class_folder"].value_counts().plot(kind="bar", title="Lung ultrasound images per class")
    plt.ylabel("Image count")
    plt.tight_layout()
    plt.show()

## Data requirements before training

- Ảnh siêu âm phổi thực, có nhãn được chuyên gia/nguồn dataset cung cấp.
- Metadata bệnh nhân hoặc study ID để chia train/validation/test theo bệnh nhân, tránh rò rỉ dữ liệu.
- Ghi rõ nhãn gốc (ví dụ A-line/B-line/consolidation hoặc normal/abnormal) và không tự suy diễn nhãn.
- Cần xác nhận quyền sử dụng, nguồn, thiết bị, view và protocol thu nhận.

**Trạng thái trong archive hiện tại:** thư mục `LUNG_ULTRASOUND/raw` trống; chưa thể train hoặc báo cáo accuracy/F1. Không dùng ECG/HRV để thay thế nhãn siêu âm phổi.